In [1]:
import pickle
from pathlib import Path
import pandas as pd

In [3]:

cache_dir_string ='/home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/'
cache_dir = Path(cache_dir_string)
trial_identity_cols = ['TaskField', 'MonkeyId', 'MonkeyName', 'MonkeyGroup']

for file in sorted(cache_dir.glob("*.pkl")):
    # Parse date and round from filename
    try:
        stem = file.stem  # e.g., "2023-09-29_round_3"
        date, round_part = stem.split('_round_')
        round_no = int(round_part)
    except Exception as e:
        print(f"Could not parse date/round from filename '{file.name}': {e}")
        continue

    # Load DataFrame
    with open(file, 'rb') as f:
        exploded_df = pickle.load(f)

    exploded_df['TrialID'] = exploded_df[trial_identity_cols].astype(str).agg('_'.join, axis=1)
    # Step 2: Group by EpochStartStop and count unique trials
    epoch_to_trial_map = exploded_df.groupby('EpochStartStop')['TrialID'].nunique().reset_index()

    # Step 3: Find EpochStartStop values shared by multiple trials (== bug!)
    duplicated_epochs = epoch_to_trial_map[epoch_to_trial_map['TrialID'] > 1]

    print(f"Found {len(duplicated_epochs)} duplicated EpochStartStop values across different trials.")

    # Optional: Extract problematic rows
    if not duplicated_epochs.empty:
        print(f"[{file.name}] Problematic EpochStartStop values:")
        print(duplicated_epochs)


Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 duplicated EpochStartStop values across different trials.
Found 0 du

In [25]:
from collections import defaultdict

# 문제 정리 저장용
all_issues = defaultdict(list)

for file in sorted(cache_dir.glob("*.pkl")):
    try:
        stem = file.stem
        date, round_part = stem.split('_round_')
        round_no = int(round_part)
    except Exception as e:
        print(f"Could not parse date/round from filename '{file.name}': {e}")
        continue

    with open(file, 'rb') as f:
        exploded_df = pickle.load(f)

    exploded_df['TrialID'] = exploded_df[trial_identity_cols].astype(str).agg('_'.join, axis=1)

    # Step 2: Find problematic epochs
    epoch_to_trial_map = exploded_df.groupby('EpochStartStop')['TrialID'].nunique().reset_index()
    duplicated_epochs = epoch_to_trial_map[epoch_to_trial_map['TrialID'] > 1]

    if not duplicated_epochs.empty:
        print(f"\n[{file.name}] ⚠ Found {len(duplicated_epochs)} duplicated EpochStartStop values across different trials.")
        bad_rows = exploded_df[exploded_df['EpochStartStop'].isin(duplicated_epochs['EpochStartStop'])]

        # Sort for clarity
        bad_rows = bad_rows.sort_values(by=['EpochStartStop', 'TrialID'])

        # Print detailed conflict info
        for epoch, group_df in bad_rows.groupby('EpochStartStop'):
            unique_trials = group_df['TrialID'].unique()
            print(f"\n  ⛔ EpochStartStop: {epoch} is shared by {len(unique_trials)} different trials:")
            for trial_id in unique_trials:
                subset = group_df[group_df['TrialID'] == trial_id]
                info = subset.iloc[0][trial_identity_cols]
                print(f"    TrialID: {trial_id} → {dict(info)}")

            # Store for further review
            all_issues[file.name].append(epoch)
